# Multimodal Product Matching — Pipeline Walkthrough

This notebook walks through every stage of the system interactively:
1. Generate & inspect sample data
2. Encode a product manually
3. Build a mini FAISS index
4. Run a full 2-stage search
5. Visualise embedding space with UMAP

In [ ]:
import sys, os
sys.path.insert(0, '..')   # project root

import torch
import numpy as np
import json
from pathlib import Path

from src.utils import load_config, get_device
cfg    = load_config('../configs/config.yaml')
device = get_device()
print('Device:', device)

## 1. Generate sample data

In [ ]:
os.chdir('..')   # run from project root
!python scripts/generate_sample_data.py --n_products 100 --n_triplets 200

In [ ]:
# Peek at catalog
with open('data/processed/catalog.jsonl') as f:
    products = [json.loads(l) for l in f]

print(f'Total products: {len(products)}')
products[:3]

## 2. Encode a single product

In [ ]:
from src.embedding import BiEncoder
import torch

model = BiEncoder(
    text_encoder_name  = cfg.model.text_encoder,
    image_encoder_name = cfg.model.image_encoder,
    fusion_dim         = cfg.model.fusion_dim,
    fusion_strategy    = cfg.model.fusion_strategy,
).to(device).eval()

print(f'Embedding dim: {model.embedding_dim}')

with torch.no_grad():
    # Query embedding (text only)
    q_emb = model.encode_query(['iPhone 14 Pro 256GB purple'], device)
    print('Query embedding shape:', q_emb.shape)
    print('L2 norm:', q_emb.norm().item())   # should be ~1.0

## 3. Build a mini FAISS index

In [ ]:
!python scripts/build_index.py \
    --catalog data/processed/catalog.jsonl \
    --output  models/faiss_index_notebook

## 4. Run end-to-end search

In [ ]:
from src.inference import MatchingPipeline

pipeline = MatchingPipeline.from_config(
    cfg, device,
    index_path = 'models/faiss_index_notebook'
)

query   = 'Tìm iPhone 14 Pro bộ nhớ 256GB màu tím'
results = pipeline.search(query)

print(f'Query: {query}\n')
for r in results:
    print(f"  [{r['rank']}] {r['title']}")
    print(f"      retrieval={r['retrieval_score']:.4f}  rerank={r['rerank_score']:.4f}")

## 5. Visualise embedding space with UMAP

In [ ]:
# pip install umap-learn matplotlib
import numpy as np
import matplotlib.pyplot as plt

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print('Install umap-learn for this cell: pip install umap-learn')

if HAS_UMAP:
    from src.retrieval import ProductIndex
    index = ProductIndex.load('models/faiss_index_notebook')
    
    # Extract raw vectors from FAISS
    n = len(index)
    vecs = index._index.reconstruct_n(0, n)   # (N, D)

    reducer = umap.UMAP(n_components=2, random_state=42)
    coords  = reducer.fit_transform(vecs)

    # Color by category
    from src.data import load_catalog
    catalog   = load_catalog('data/processed/catalog.jsonl')
    cats      = [catalog.get(pid, {}).get('category', 'unknown') for pid in index.id_map]
    cat_set   = sorted(set(cats))
    colors    = [cat_set.index(c) for c in cats]

    plt.figure(figsize=(9, 7))
    scatter = plt.scatter(coords[:, 0], coords[:, 1], c=colors, cmap='tab10', alpha=0.6, s=15)
    plt.colorbar(scatter, ticks=range(len(cat_set)), label='Category')
    plt.clim(-0.5, len(cat_set) - 0.5)
    cb = plt.gcf().axes[-1]
    cb.set_yticklabels(cat_set)
    plt.title('Product Embedding Space (UMAP)')
    plt.tight_layout()
    plt.savefig('notebooks/embeddings_umap.png', dpi=150)
    plt.show()

## 6. Evaluate metrics

In [ ]:
!python scripts/evaluate.py \
    --index   models/faiss_index_notebook \
    --val     data/processed/val.jsonl \
    --stage   1